# Explore MSD Datasets

This notebook uses the standard PyHealth flow: dataset → `set_task()` → `SampleDataset` → `get_dataloader()`. `show_msd_samples()` can either show image and mask separately or overlay the mask on the image. Data and caches default to `~/pyhealth-imaging-data` and `~/.cache/pyhealth-imaging` (`PYHEALTH_IMAGING_DATA` / `PYHEALTH_IMAGING_CACHE` override them); `download=True` fetches missing MSD tasks automatically.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from pyhealth.datasets import (
    MSDHippocampusDataset,
    MSDProstateDataset,
    MSDSpleenDataset,
    get_dataloader,
)

# Override these with environment variables if your data/cache live elsewhere:
#   PYHEALTH_IMAGING_DATA
#   PYHEALTH_IMAGING_CACHE
DATA_ROOT = Path(
    os.environ.get("PYHEALTH_IMAGING_DATA", "~/pyhealth-imaging-data")
).expanduser()
CACHE_DIR = Path(
    os.environ.get("PYHEALTH_IMAGING_CACHE", "~/.cache/pyhealth-imaging")
).expanduser()
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("TORCH_HOME", str(CACHE_DIR / "torch"))
print("Data directory:", DATA_ROOT)
print("Cache directory:", CACHE_DIR)

# Processed caches and generated metadata are stored here, outside the repo.
METADATA_DIR = CACHE_DIR / "metadata"


DATASETS = {
    "hippocampus": (MSDHippocampusDataset, DATA_ROOT / "Task04_Hippocampus"),
    "prostate": (MSDProstateDataset, DATA_ROOT / "Task05_Prostate"),
    "spleen": (MSDSpleenDataset, DATA_ROOT / "Task09_Spleen"),
}


def load_msd_samples(name):
    DatasetClass, root = DATASETS[name]
    dataset = DatasetClass(
        root=root,
        cache_dir=CACHE_DIR,
        metadata_dir=str(Path(METADATA_DIR) / name),
        download=True,
    )
    dataset.stats()
    samples = dataset.set_task(dataset.default_task, num_workers=1)
    sample = samples[0]
    image = samples.input_processors["image"].materialize(sample["image"])
    mask = samples.output_processors["mask"].materialize(sample["mask"])
    print("samples:", len(samples))
    print("input schema:", samples.input_schema)
    print("output schema:", samples.output_schema)
    print("image:", image.shape, image.dtype)
    print("mask:", mask.shape, mask.dtype)
    return dataset, samples


def show_msd_samples(samples, n=3, overlay_mask=False, channel=0):
    n = min(n, len(samples))
    rows = 1 if overlay_mask else 2
    fig, axes = plt.subplots(rows, n, figsize=(4 * n, 4 * rows), squeeze=False)
    for col in range(n):
        sample = samples[col]
        image_vol = samples.input_processors["image"].materialize(sample["image"])
        mask_vol = samples.output_processors["mask"].materialize(sample["mask"])
        slice_idx = image_vol.shape[1] // 2
        image = image_vol[channel, slice_idx].cpu()
        mask = mask_vol[0, slice_idx].cpu()
        channel_label = f" ch{channel}" if image_vol.shape[0] > 1 else ""
        if overlay_mask:
            axes[0, col].imshow(image, cmap="gray")
            axes[0, col].imshow(
                np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.35
            )
            axes[0, col].set_title(f"{sample['patient_id']}{channel_label}")
        else:
            axes[0, col].imshow(image, cmap="gray")
            axes[0, col].set_title(f"{sample['patient_id']}{channel_label}")
            axes[1, col].imshow(mask, cmap="gray")
            axes[1, col].set_title("mask")
        for row in range(rows):
            axes[row, col].axis("off")
    plt.show()


## Hippocampus

Mask is overlaid on the image.

In [ ]:
hippo_dataset, hippo_samples = load_msd_samples("hippocampus")
show_msd_samples(hippo_samples, n=3, overlay_mask=True)


In [ ]:
loader = get_dataloader(hippo_samples, batch_size=1)
batch = next(iter(loader))
print("batch image:", batch["image"].shape)
print("batch mask:", batch["mask"].shape)


## Prostate

Prostate has two channels (T2, ADC); the plot uses channel 0.

In [ ]:
prostate_dataset, prostate_samples = load_msd_samples("prostate")
show_msd_samples(prostate_samples, n=3, overlay_mask=True, channel=0)


## Spleen

Mask is overlaid on the CT image.

In [ ]:
spleen_dataset, spleen_samples = load_msd_samples("spleen")
show_msd_samples(spleen_samples, n=3, overlay_mask=True)


Set `overlay_mask=False` in any `show_msd_samples()` call to display the image and mask as separate rows instead.

## Segmentation training (3D U-Net)

This section runs a normal multi-epoch training job on the Hippocampus training split, using the full split by default. `UNet3D` uses the original U-Net paper defaults (64 base channels, depth 5, no normalisation, up-convolutions) for every dataset.

For Spleen, compute the CT foreground statistics on the training data and pass them to the image processor:

```python
from pyhealth.datasets import PatientSplit, compute_foreground_intensity_stats
from pyhealth.processors import NiftiProcessor
from pyhealth.tasks import MSDSegmentationTask

spleen_raw = spleen_dataset.set_task(
    MSDSegmentationTask(normalize=None, num_classes=spleen_dataset.num_classes),
    num_workers=1,
)
ct_stats = compute_foreground_intensity_stats(spleen_raw, modality="CT")
spleen_train, spleen_val = spleen_dataset.set_task(
    spleen_dataset.default_task,
    split=PatientSplit((0.8, 0.2), seed=42),
    input_processors={"image": NiftiProcessor(normalize="ct", intensity_stats=ct_stats)},
)
```


**Cache note:** `litdata>=0.2.71` uses memory-mapped chunk readers for map-style indexing, and `PatchDataset` materializes lazy NIfTI paths per DataLoader worker so workers do not share open file handles.

The 3D U-Net training below uses MONAI's Dice loss and sliding-window inference. Install the imaging extra with `pip install 'pyhealth[imaging]'` (MONAI is included).


In [ ]:
import numpy as np
import torch

from pyhealth.datasets import PatientSplit, PatchDataset, get_dataloader
from pyhealth.models import UNet3D, evaluate_segmentation
from pyhealth.sampler import PatchSampler
from pyhealth.trainer import Trainer

torch.backends.cudnn.benchmark = True

EPOCHS = 30
PATIENCE = 10
BATCH_SIZE = 8
NUM_WORKERS = 4
PIN_MEMORY = torch.cuda.is_available()
PATCH_SIZE = (32, 32, 32)
MAX_TRAIN_CASES = None  # set an integer for a quicker run
MAX_VAL_CASES = None

device = "cuda" if torch.cuda.is_available() else "cpu"

train_ds, val_ds = hippo_dataset.set_task(
    hippo_dataset.default_task,
    split=PatientSplit(ratios=(0.8, 0.2), seed=42),
    num_workers=1,
)
if MAX_TRAIN_CASES is not None and MAX_TRAIN_CASES < len(train_ds):
    rng = np.random.default_rng(42)
    train_ds = train_ds.subset(
        rng.choice(len(train_ds), MAX_TRAIN_CASES, replace=False)
    )
if MAX_VAL_CASES is not None and MAX_VAL_CASES < len(val_ds):
    rng = np.random.default_rng(43)
    val_ds = val_ds.subset(
        rng.choice(len(val_ds), MAX_VAL_CASES, replace=False)
    )

train_patches = PatchDataset(
    train_ds,
    PatchSampler(PATCH_SIZE, foreground_prob=1 / 3),
    mode="train",
)
val_patches = PatchDataset(
    val_ds,
    PatchSampler(PATCH_SIZE),
    mode="center",
)
loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
if NUM_WORKERS > 0:
    loader_kwargs.update(persistent_workers=True, prefetch_factor=2)
train_loader = get_dataloader(
    train_patches, batch_size=BATCH_SIZE, shuffle=True, **loader_kwargs
)
val_loader = get_dataloader(
    val_patches, batch_size=BATCH_SIZE, **loader_kwargs
)

model = UNet3D(
    train_ds,
    in_channels=1,
    num_classes=hippo_dataset.num_classes,
)
print("parameters (M):", sum(p.numel() for p in model.parameters()) / 1e6)

trainer = Trainer(model, device=device, metrics=["dice", "iou"])
trainer.train(
    train_loader,
    val_dataloader=val_loader,
    epochs=EPOCHS,
    optimizer_params={"lr": 5e-4},
    weight_decay=1e-4,
    max_grad_norm=1.0,
    monitor="dice",
    monitor_criterion="max",
    patience=PATIENCE,
)

print("patch-level validation:", trainer.evaluate(val_loader))

from pyhealth.metrics import per_class_segmentation_metrics

y_true, y_prob, _ = trainer.inference(val_loader)
per_class = per_class_segmentation_metrics(
    y_true,
    y_prob,
    mode=model.mode,
    num_classes=hippo_dataset.num_classes,
    ignore_index=getattr(model, "segmentation_ignore_index", None),
)
print("per-class validation Dice:", per_class)

print(
    "case-level validation:",
    evaluate_segmentation(
        model, val_ds, patch_size=PATCH_SIZE, overlap=0.5, batch_size=8
    ),
)

run_dir = CACHE_DIR / "runs"
run_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), run_dir / "msd_hippocampus_unet3d.ckpt")
